# DeepSequence v1.6 — example notebook

End-to-end intermittent demand forecast with **feature contract v1.6** (28 columns: holiday **distance only**, no binary `is_*`).

This notebook builds a **synthetic** panel so it runs without external data. For confidentiality and reproducibility notes, see **Dataset Availability** in `README.md` and the aggregated bake-off summary in `REPORT_v1.6.md`.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf

ROOT = Path.cwd().resolve()
if (ROOT / "deepsequence_hierarchical_attention").is_dir() and (ROOT / "examples").is_dir():
    PKG_ROOT = ROOT
elif (ROOT.parent / "deepsequence_hierarchical_attention").is_dir():
    PKG_ROOT = ROOT.parent
else:
    PKG_ROOT = ROOT

sys.path.insert(0, str(PKG_ROOT))
sys.path.insert(0, str(PKG_ROOT / "examples"))

from deepsequence_hierarchical_attention import (
    __version__,
    build_hierarchical_model_lightweight,
    get_feature_config_path,
    round_forecast,
)
from feature_config_loader import load_feature_config
from train_lightweight_adaptive_loss import AdaptiveWeightedModel, WeightedBCELoss

tf.keras.utils.set_random_seed(42)
print("package", __version__)
print("feature_config", get_feature_config_path())
assert __version__ == "1.6.0"

## 1. Synthetic intermittent panel + holiday distances

In [ ]:
rng = np.random.default_rng(42)
n_skus, n_days = 12, 90
skus = [f"SKU_{i:03d}" for i in range(n_skus)]
dates = pd.date_range("2024-01-01", periods=n_days, freq="D")

rows = []
for sku in skus:
    rate = rng.uniform(0.05, 0.25)
    mag = rng.uniform(1.0, 8.0)
    for ds in dates:
        y = float(rng.poisson(mag)) if rng.random() < rate else 0.0
        rows.append({"id_var": sku, "ds": ds, "Quantity": y})
panel = pd.DataFrame(rows)

cfg = load_feature_config()
assert cfg.config["metadata"]["version"] == "1.6"
assert cfg.total_features == 28
assert cfg.binary_holiday_names == []

hol = pd.DataFrame(
    {name: rng.integers(-30, 60, size=len(panel)).astype(float) for name in cfg.holiday_names}
)

print("rows", len(panel), "zero_rate", float((panel["Quantity"] == 0).mean()))
print("n_features", cfg.total_features, cfg.feature_names)

## 2. Causal features (v1.6) and train / val split

In [ ]:
split_day = dates[int(n_days * 0.75)]
train_df = panel.loc[panel["ds"] < split_day].reset_index(drop=True)
val_df = panel.loc[panel["ds"] >= split_day].reset_index(drop=True)
h_tr = hol.loc[panel["ds"] < split_day].reset_index(drop=True)
h_va = hol.loc[panel["ds"] >= split_day].reset_index(drop=True)

Xtr_df, states = cfg.create_features(train_df, h_tr, return_states=True)
Xva_df, _ = cfg.create_features(val_df, h_va, prior_states=states, return_states=True)
assert list(Xtr_df.columns) == cfg.feature_names
assert Xtr_df.shape[1] == 28
assert "lag_1" in Xtr_df.columns and "is_NewYear" not in Xtr_df.columns

X_train = Xtr_df.to_numpy(np.float32)
X_val = Xva_df.to_numpy(np.float32)
t_idx = cfg.trend_indices[0]
tmin, tmax = float(X_train[:, t_idx].min()), float(X_train[:, t_idx].max())
span = max(tmax - tmin, 1.0)
X_train[:, t_idx] = (X_train[:, t_idx] - tmin) / span
X_val[:, t_idx] = (X_val[:, t_idx] - tmin) / span

cats = pd.Categorical(train_df["id_var"])
sku_map = {k: i for i, k in enumerate(cats.categories)}
n_skus = len(sku_map)

def enc(df):
    return df["id_var"].map(sku_map).astype(np.int32).to_numpy().reshape(-1, 1)

y_train = train_df["Quantity"].to_numpy(np.float32)
y_val = val_df["Quantity"].to_numpy(np.float32)
sku_train, sku_val = enc(train_df), enc(val_df)
zero_rate = float((y_train == 0).mean())

def split_components(X):
    return (
        X[:, cfg.trend_indices].astype(np.float32),
        X[:, cfg.seasonal_indices].astype(np.float32),
        X[:, cfg.holiday_indices].astype(np.float32),
        X[:, cfg.regressor_indices].astype(np.float32),
    )

tr, va = split_components(X_train), split_components(X_val)
print("train/val", len(y_train), len(y_val), "zero_rate", round(zero_rate, 3))
print("holiday dim", tr[2].shape[1], "(distance only)")

## 3. Build & train DeepSequence (gated)

In [ ]:
base = build_hierarchical_model_lightweight(
    n_temporal_features=len(cfg.trend_indices),
    n_fourier_features=len(cfg.seasonal_indices),
    n_holiday_features=len(cfg.holiday_indices),
    n_lag_features=len(cfg.regressor_indices),
    n_skus=n_skus,
    hidden_dim=32,
    sku_embedding_dim=4,
    dropout_rate=0.1,
    use_cross_layers=True,
    use_intermittent=True,
    n_changepoints=8,
)
_ = base(
    [*(np.zeros((1, x.shape[1]), np.float32) for x in tr), np.zeros((1, 1), np.int32)],
    training=False,
)

pos_weight = min(20.0, zero_rate / max(1.0 - zero_rate, 1e-3))
model = AdaptiveWeightedModel(
    base_model=base,
    bce_loss_fn=WeightedBCELoss(weight_nonzero=pos_weight, weight_zero=1.0),
    mae_loss_fn=tf.keras.losses.MeanAbsoluteError(),
    zero_rate=zero_rate,
    avg_nonzero_demand=float(y_train[y_train > 0].mean()) if (y_train > 0).any() else 1.0,
    pos_weight=pos_weight,
    loss_recipe="three_term",
    alpha_bce=0.2,
    w_gated=1.0,
    w_mag=1.0,
    use_fixed_weights=True,
)
model.compile(optimizer=tf.keras.optimizers.Adam(0.0025))

ytr = {"final_forecast": y_train.reshape(-1, 1), "base_forecast": y_train.reshape(-1, 1)}
yva = {"final_forecast": y_val.reshape(-1, 1), "base_forecast": y_val.reshape(-1, 1)}

history = model.fit(
    [*tr, sku_train],
    ytr,
    validation_data=([*va, sku_val], yva),
    epochs=3,
    batch_size=256,
    verbose=2,
)
assert len(history.history["loss"]) == 3
print("final train loss", history.history["loss"][-1])

## 4. Predict & quick metrics

In [ ]:
pred = model.predict([*va, sku_val], batch_size=512, verbose=0)
yhat = np.asarray(pred["final_forecast"]).reshape(-1)
p = np.asarray(pred["non_zero_probability"]).reshape(-1)
yhat_r = round_forecast(yhat)

mae = float(np.mean(np.abs(y_val - yhat_r)))
nz = y_val > 0
mae_nz = float(np.mean(np.abs(y_val[nz] - yhat_r[nz]))) if nz.any() else float("nan")
bias = float(np.mean(yhat_r - y_val))

print(f"val MAE(rounded)={mae:.3f}  nonzero MAE={mae_nz:.3f}  bias={bias:.3f}")
print(f"p mean={float(p.mean()):.3f}  p in (0,1) ok={bool(((p > 0) & (p < 1)).mean() > 0.5)}")
assert np.isfinite(mae)
assert yhat.shape == y_val.shape
assert ((p >= 0) & (p <= 1)).all()
print("OK — v1.6 example completed.")